# Manifest Alchemy — FLUX.1 LoRA Training
**Trains your MANIFESTA aesthetic on FLUX.1-dev using ai-toolkit**

## Before you run this notebook:
1. Create a free account at **huggingface.co**
2. Go to **huggingface.co/black-forest-labs/FLUX.1-dev** → click Agree to accept terms
3. Go to **huggingface.co/settings/tokens** → New Token → Read access → copy it
4. In Kaggle: go to **Add-ons → Secrets** → add secret named `HF_TOKEN` with your token value
5. Upload your `flux_dataset.zip` as a Kaggle Dataset named **manifesta-flux-dataset**
6. Add the dataset to this notebook (right panel → Input → + Add → Your Datasets)
7. Turn on GPU: Settings → Accelerator → **GPU T4 x2** (or T4 x1)
8. Run all cells in order

In [ ]:
# Cell 1 — Check GPU
import subprocess
result = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'], 
                       capture_output=True, text=True)
print('GPU:', result.stdout.strip())

import torch
print('CUDA available:', torch.cuda.is_available())
print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')

In [ ]:
# Cell 2 — Clone ai-toolkit (the best free FLUX LoRA trainer)
import os
os.chdir('/kaggle/working')

!git clone https://github.com/ostris/ai-toolkit.git
os.chdir('/kaggle/working/ai-toolkit')
!git submodule update --init --recursive

In [ ]:
# Cell 3 — Install dependencies (takes ~3 minutes)
!pip install -r requirements.txt -q
!pip install bitsandbytes -q
print('Done installing.')

In [ ]:
# Cell 4 — Login to HuggingFace with your secret token
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login

secrets = UserSecretsClient()
hf_token = secrets.get_secret('HF_TOKEN')
login(token=hf_token, add_to_git_credential=False)
print('Logged in to HuggingFace.')

In [ ]:
# Cell 5 — Extract dataset
import zipfile, os, shutil

# Look for the uploaded dataset
DATASET_INPUT = '/kaggle/input/manifesta-flux-dataset'
IMAGES_DIR = '/kaggle/working/flux_dataset/images'

if not os.path.exists(IMAGES_DIR):
    # Find the zip file
    zip_path = None
    for root, dirs, files in os.walk(DATASET_INPUT):
        for f in files:
            if f.endswith('.zip'):
                zip_path = os.path.join(root, f)
                break
    
    if zip_path:
        print(f'Extracting {zip_path}...')
        with zipfile.ZipFile(zip_path, 'r') as z:
            z.extractall('/kaggle/working/')
    else:
        # Dataset was already extracted by Kaggle
        shutil.copytree(DATASET_INPUT, '/kaggle/working/flux_dataset', dirs_exist_ok=True)

# Verify
img_count = len([f for f in os.listdir(IMAGES_DIR) if f.endswith('.png')])
print(f'Found {img_count} images ready for training.')

In [ ]:
# Cell 6 — Merge captions into image folder
# ai-toolkit expects captions (.txt) in the SAME folder as images
import shutil, os

CAPTIONS_DIR = '/kaggle/working/flux_dataset/captions'
IMAGES_DIR = '/kaggle/working/flux_dataset/images'

moved = 0
for fname in os.listdir(CAPTIONS_DIR):
    if fname.endswith('.txt'):
        src = os.path.join(CAPTIONS_DIR, fname)
        dst = os.path.join(IMAGES_DIR, fname)
        if not os.path.exists(dst):
            shutil.copy2(src, dst)
            moved += 1

print(f'Copied {moved} caption files into images folder.')

# Quick sanity check — print first caption
first_txt = sorted([f for f in os.listdir(IMAGES_DIR) if f.endswith('.txt')])[0]
with open(os.path.join(IMAGES_DIR, first_txt)) as f:
    print('Sample caption:', f.read()[:120])

In [ ]:
# Cell 7 — Write training config
import yaml, os

config = {
    'job': 'extension',
    'config': {
        'name': 'manifesta-flux-lora',
        'process': [{
            'type': 'sd_trainer',
            'training_folder': '/kaggle/working/output',
            'device': 'cuda:0',
            'trigger_word': 'MANIFESTA',
            'network': {
                'type': 'lora',
                'linear': 16,
                'linear_alpha': 16
            },
            'save': {
                'dtype': 'float16',
                'save_every': 500,
                'max_step_saves_to_keep': 3
            },
            'datasets': [{
                'folder_path': '/kaggle/working/flux_dataset/images',
                'caption_ext': 'txt',
                'caption_dropout_rate': 0.05,
                'shuffle_tokens': False,
                'cache_latents_to_disk': True,
                'resolution': [512, 768, 1024]
            }],
            'train': {
                'batch_size': 1,
                'steps': 2500,
                'gradient_accumulation_steps': 1,
                'train_unet': True,
                'train_text_encoder': False,
                'gradient_checkpointing': True,
                'noise_scheduler': 'flowmatch',
                'optimizer': 'adamw8bit',
                'lr': 0.0004,
                'ema_config': {
                    'use_ema': True,
                    'ema_decay': 0.99
                },
                'dtype': 'bf16'
            },
            'model': {
                'name_or_path': 'black-forest-labs/FLUX.1-dev',
                'is_flux': True,
                'quantize': True  # Quantize to fit in 16 GB VRAM
            },
            'sample': {
                'sampler': 'flowmatch',
                'sample_every': 500,
                'width': 1024,
                'height': 1024,
                'prompts': [
                    'MANIFESTA, a woman receiving abundance, golden light, cinematic, divine feminine',
                    'MANIFESTA, luxury manifestation scene, mystical golden energy, opulent',
                    'MANIFESTA, aspirational lifestyle, golden hour, high vibrational energy'
                ],
                'neg': '',
                'seed': 42,
                'walk_seed': True,
                'guidance_scale': 3.5,
                'sample_steps': 28
            }
        }]
    },
    'meta': {
        'name': 'manifesta-flux-lora',
        'version': '1.0'
    }
}

os.makedirs('/kaggle/working/ai-toolkit/config/custom', exist_ok=True)
config_path = '/kaggle/working/ai-toolkit/config/custom/manifesta.yaml'
with open(config_path, 'w') as f:
    yaml.dump(config, f, default_flow_style=False)

print('Config written to:', config_path)
print('Training steps: 2500 | Trigger word: MANIFESTA | Images:', 
      len([f for f in os.listdir('/kaggle/working/flux_dataset/images') if f.endswith('.png')]))

In [ ]:
# Cell 8 — START TRAINING
# This takes ~2-4 hours on T4. Do not close the tab.
# Every 500 steps you'll see sample images generated with MANIFESTA trigger.
# Watch the loss — it should go from ~1.0 down to ~0.1-0.3

os.chdir('/kaggle/working/ai-toolkit')
!python run.py config/custom/manifesta.yaml

In [ ]:
# Cell 9 — Find and display your trained LoRA file
import os

output_dir = '/kaggle/working/output/manifesta-flux-lora'
lora_files = []
for root, dirs, files in os.walk(output_dir):
    for f in files:
        if f.endswith('.safetensors'):
            full = os.path.join(root, f)
            size_mb = os.path.getsize(full) / 1e6
            lora_files.append((full, size_mb))

if lora_files:
    # Show the final model (largest file or last saved)
    final = sorted(lora_files, key=lambda x: x[1], reverse=True)[0]
    print(f'Your trained LoRA: {final[0]}')
    print(f'Size: {final[1]:.1f} MB')
    print()
    print('Download this file from Kaggle Output panel (right side → Output tab)')
    print('Then upload to Replicate to use it in your app.')
else:
    print('No .safetensors found. Check training output above for errors.')

In [ ]:
# Cell 10 — Preview sample images generated during training
from IPython.display import Image, display
import os, glob

sample_dir = '/kaggle/working/output/manifesta-flux-lora'
samples = sorted(glob.glob(f'{sample_dir}/**/*.jpg', recursive=True) + 
                 glob.glob(f'{sample_dir}/**/*.png', recursive=True))

# Show last 3 sample images (most recent = best quality)
for s in samples[-3:]:
    print(os.path.basename(s))
    display(Image(s, width=512))